# Garbage Classification — Full Experiment Pipeline on Google Colab

**Full workflow** theo `experiments/EXPERIMENTS_GUIDE.md`, chạy trên GPU miễn phí.

**Bước 0**: Đặt runtime thành GPU
- Nhấp **Runtime** → **Change runtime type** → GPU → **Save**

## 1. Setup Environment

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

In [ ]:
from pathlib import Path
import os

drive_root = Path('/content/drive/MyDrive')
project_candidates = sorted(drive_root.glob('Garbage_Classification*'))

if not project_candidates:
    raise FileNotFoundError('Khong tim thay folder Garbage_Classification* trong MyDrive')

DRIVE_PROJECT_DIR = str(project_candidates[0])
DATA_DIR = f'{DRIVE_PROJECT_DIR}/data'
PROCESSED_DIR = f'{DATA_DIR}/processed'

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f'Khong tim thay folder data tren Drive: {DATA_DIR}')

print(f'✓ Using Drive data folder: {DATA_DIR}')
print(f'✓ Using processed data folder: {PROCESSED_DIR}')
print('Top-level entries:')
for name in sorted(os.listdir(DATA_DIR))[:20]:
    print(f'- {name}')

### 1.2 Clone Repository

In [ ]:
# Neu repo da ton tai thi chi cd vao, tranh clone long nhau
import os

if os.path.isdir('Garbage_Classification_Project') and os.path.isfile('Garbage_Classification_Project/README.md'):
    %cd Garbage_Classification_Project
    print("✓ Repo already exists, reused current copy")
else:
    !git clone https://github.com/dinhhoang235/Garbage_Classification_Project.git
    %cd Garbage_Classification_Project
    print("✓ Repo cloned")

# Neu private: upload zip vao Drive, sau do uncomment va chay
# !cp /content/drive/MyDrive/Garbage_Classification_Project.zip .
# !unzip -q Garbage_Classification_Project.zip
# %cd Garbage_Classification_Project

### 1.3 Check GPU & Install Dependencies

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q -r requirements.txt
print("✓ Dependencies installed")

### 1.4 Use data folder directly from Google Drive

In [ ]:
# Dung luon folder data da co san tren Drive, khong copy lai vao local data/
!find "$DATA_DIR" -maxdepth 2 -type d | head -50
# Neu ban da co san data/processed tren Drive, co the bo qua buoc preprocess ben duoi.

## 2. Prepare Data (Optional if `data/processed` already exists)

Neu ban da co san `data/processed` tren Drive, co the bo qua toan bo muc nay.
Chi chay lai khi muon tao moi split/processed tu `data/raw/original` bang `src/preprocess.py`.

In [ ]:
from pathlib import Path

raw_original = Path(f'{DATA_DIR}/raw/original')
processed_root = Path(PROCESSED_DIR)
train_dir = processed_root / 'train'

# Neu da co san `data/processed/train` tren Drive, cell nay chi can kiem tra va co the bo qua preprocessing.
# Chi khi chua co processed data moi can chay lai `src/preprocess.py` de tao `data/processed`.
if train_dir.exists():
    print(f'✓ Found processed data at {processed_root}')
else:
    if not raw_original.exists():
        raise FileNotFoundError(f'Neither processed data nor raw/original found. Missing raw data: {raw_original}')

    print('Running preprocessing pipeline from src/preprocess.py ...')
    !python3 src/preprocess.py --base_dir "$raw_original" --processed_dir "$processed_root"
    print(f'✓ Preprocessed data saved to {processed_root}')

print('Processed directory tree:')
!find "$PROCESSED_DIR" -maxdepth 2 -type d | sort

## 3. Run Batch Experiments

Chạy batch training: 3 configs × 30 epochs mỗi

### Batch run: 3 models × 30 epochs

In [ ]:
print('Chay FULL EXPERIMENTS: 3 configs x 30 epochs')
print('='*60)
print('Models: MobileNetV1, MobileNetV3Large, EfficientNetV2S')
print(f'Base data dir: {PROCESSED_DIR}')
print('Thoi gian du tinh: ~2-3 gio (tuy GPU)\n')

In [ ]:

from pathlib import Path
from subprocess import run

# Verify processed data directory structure before running experiments
processed_root = Path(PROCESSED_DIR)
if not processed_root.exists():
    raise FileNotFoundError(f"Processed directory not found: {PROCESSED_DIR}")

train_dir = processed_root / 'train'
if not train_dir.exists():
    raise FileNotFoundError(f"Expected processed data not found: {train_dir}")

# Count classes and images
classes = [d for d in train_dir.iterdir() if d.is_dir()]
print(f"✓ Found {len(classes)} classes in {train_dir}")

for cls_dir in sorted(classes)[:5]:  # Show first 5 classes
    img_count = len(list(cls_dir.glob('*.*')))
    print(f"  - {cls_dir.name}: {img_count} images")

if len(classes) == 0:
    raise RuntimeError(f"No class folders found in {train_dir}. Check data structure.")

print(f"\n✓ Data directory structure is valid. Ready to run experiments.")


In [ ]:
# Copy processed data to local Colab storage for faster I/O (30-40% speedup)
import shutil
import os
from pathlib import Path

local_processed_dir = '/content/local_data'
local_train_dir = os.path.join(local_processed_dir, 'train')

if os.path.exists(local_processed_dir):
    print(f"✓ Local data folder already exists at {local_processed_dir}")
else:
    print(f"Copying data from Drive to local storage...")
    print(f"Source: {PROCESSED_DIR}")
    print(f"Destination: {local_processed_dir}")
    
    shutil.copytree(PROCESSED_DIR, local_processed_dir)
    
    # Verify
    num_classes = len([d for d in Path(local_train_dir).iterdir() if d.is_dir()])
    total_images = sum(len(list(Path(local_train_dir).glob('*/*'))))
    
    print(f"✓ Copy completed!")
    print(f"✓ Found {num_classes} classes")
    print(f"✓ Total images: {total_images}")
    print(f"✓ Local data ready. Training will use local disk (~2-3x faster than Drive)")
    
    # Update configs to use local path instead of Drive
    for config_path in Path('experiments/configs').glob('*.yaml'):
        config_text = config_path.read_text(encoding='utf-8')
        config_text = config_text.replace(f'base_dir: {PROCESSED_DIR}', 'base_dir: /content/local_data')
        config_text = config_text.replace('base_dir: data/processed', 'base_dir: /content/local_data')
        config_path.write_text(config_text, encoding='utf-8')
    
    print(f"✓ Updated all configs to use local data path")

In [ ]:
# Batch run: tat ca 3 models (stream log realtime, unbuffered)
# NOTE: configs đã được update bởi cell copy local, nên không truyền --base_dir
import subprocess

cmd = [
    'python3', '-u',
    'tools/run_experiments.py',
    '--configs',
    'experiments/configs/mobilenetv1_baseline.yaml',
    'experiments/configs/mobilenetv3_large.yaml',
    'experiments/configs/efficientnetv2s.yaml',
]

print('Starting batch experiments...')
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
 )

for line in proc.stdout:
    print(line, end='')

return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(f'Experiment script failed with exit code {return_code}')

print('✓ All experiments completed successfully')


### Troubleshooting

Nếu gặp lỗi, thử các bước sau:

1. **Verify data exists**: Chạy cell trước (data validation) để xác nhận dữ liệu
2. **Check GPU memory**: `!nvidia-smi` - nếu OOM, giảm batch_size
3. **Install missing packages**: `!pip install --upgrade tensorflow scikit-learn`
4. **Check file paths**: Xác nhận DATA_DIR trỏ đúng folder dữ liệu
5. **Run single experiment**: Test một config trước batch run để debug dễ hơn


In [ ]:
# Copy toàn bộ results về Drive
!mkdir -p /content/drive/MyDrive/garbage_runs/full_experiments
!cp -r model/weights /content/drive/MyDrive/garbage_runs/full_experiments/
!cp -r experiments/results_summary.csv /content/drive/MyDrive/garbage_runs/full_experiments/ 2>/dev/null || echo "Results file updated"
!cp -r experiments/plots /content/drive/MyDrive/garbage_runs/full_experiments/ 2>/dev/null || echo "Plots folder pending"
print("✓ Results & checkpoints lưu vào Drive")

## 4. Evaluate Models on Test Set

Sinh classification report và confusion matrix cho từng model

In [ ]:
# Đánh giá từng model lưu trong model/weights/
import os
import glob

models = glob.glob('model/weights/*_best.keras')
print(f"Tìm thấy {len(models)} model(s)")
for m in models:
    print(f"  - {m}")

In [ ]:
# Evaluate tung model tren PROCESSED_DIR
from subprocess import run

for model_path in models:
    print(f'\nEvaluating {model_path}...')
    run([
        'python3',
        'src/evaluate.py',
        '--model_path', model_path,
        '--base_dir', PROCESSED_DIR,
        '--img_size', '224', '224',
        '--batch_size', '32',
    ], check=True)

## 5. Generate Comparison Plots

In [ ]:
# Vẽ biểu đồ so sánh (nếu có results_summary.csv)
from pathlib import Path
from subprocess import run

results_path = Path('experiments/results_summary.csv')
if results_path.exists():
    run(['python3', 'tools/plot_results.py', '--results_path', str(results_path)], check=True)
    print("✓ Plots generated → experiments/plots/")
else:
    print("⚠ results_summary.csv chưa có. Chạy batch run (Cell 3B) trước.")

## 6. View Training History (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir . --port 6006

## 7. Save Everything to Drive

In [ ]:
# Backup final results & plots
!mkdir -p /content/drive/MyDrive/garbage_runs/final_results

# Copy results
!cp -r experiments/results_summary.csv /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null
!cp -r experiments/plots /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null
!cp -r reports /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null

print("✓ All results saved to Drive: MyDrive/garbage_runs/final_results")

## 8. Quick Analysis

In [ ]:
# Đọc và hiển thị results summary
import pandas as pd

try:
    df = pd.read_csv('experiments/results_summary.csv')
    print("📊 Results Summary:")
    print(df)
    
    # Tìm best model
    best_idx = df['test_accuracy'].idxmax()
    print(f"\n🏆 Best model: {df.loc[best_idx, 'backbone']} (Accuracy: {df.loc[best_idx, 'test_accuracy']:.4f})")
except:
    print("⚠ Chưa có results_summary.csv")

## Tips

- ✓ Chạy batch experiments: 3 models × 30 epochs (~2-3 giờ)
- ✓ Luôn backup kết quả vào Drive trước khi session timeout
- ✓ Kiểm tra GPU memory với `nvidia-smi` nếu OOM
- ✓ Dùng Colab Pro nếu cần chạy lâu (Colab miễn phí giới hạn ~12h)